# 01 — Data audit

**Phase 0 · MetaFlex Decoded**

The point of this notebook is not to clean anything. It is to find out what is actually
in these files, so that every cleaning decision later is a decision and not a guess.

By the end I should be able to answer:

1. How many patients, how many patient-visits, how many files?
2. What are the columns, in both layers, and what do they mean?
3. How much data is missing, and where?
4. Are the timestamps regular? Where are the gaps?
5. Which files are `.xls` and which are `.xlsx`, and does it matter?
6. What surprised me?

Anything I decide here gets written into `docs/data-decisions.md`.

## Setup

In [2]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")     # notebook lives in notebooks/, data lives in data/raw/

print("Looking in:", RAW.resolve())
print("Exists:", RAW.exists())


Looking in: /Users/sammixie/Desktop/MetaFlex_decoded/data/raw
Exists: True


### If that printed `Exists: False`

The path is relative to where this notebook file sits. If your notebook is in
`metaflex-decoded/notebooks/` and the data is in `metaflex-decoded/data/raw/`, then
`../data/raw` is correct. If not, adjust `RAW` until `Exists: True`.

Do not move on until it is True.

## 1. Inventory — what files do I actually have?

In [ ]:
# List everything under data/raw, whatever the folder structure turns out to be.
all_files = sorted(RAW.rglob("*"))
files = [f for f in all_files if f.is_file()]

print(f"Total files: {len(files)}\n")

# Group by extension
from collections import Counter
ext_counts = Counter(f.suffix.lower() for f in files)
print("By extension:")
for ext, n in ext_counts.most_common():
    print(f"  {ext or '(none)':10s} {n}")

print("\nBy folder:")
folder_counts = Counter(str(f.parent.relative_to(RAW)) for f in files)
for folder, n in sorted(folder_counts.items()):
    print(f"  {folder:30s} {n}")

[PosixPath('../data/raw/.DS_Store'), PosixPath('../data/raw/Shanghai_T1DM'), PosixPath('../data/raw/Shanghai_T1DM/1001_0_20210730.xlsx'), PosixPath('../data/raw/Shanghai_T1DM/1002_0_20210504.xls'), PosixPath('../data/raw/Shanghai_T1DM/1002_1_20210521.xls'), PosixPath('../data/raw/Shanghai_T1DM/1002_2_20210909.xls'), PosixPath('../data/raw/Shanghai_T1DM/1003_0_20210831.xls'), PosixPath('../data/raw/Shanghai_T1DM/1004_0_20210425.xls'), PosixPath('../data/raw/Shanghai_T1DM/1005_0_20210522.xls'), PosixPath('../data/raw/Shanghai_T1DM/1006_0_20210114.xlsx'), PosixPath('../data/raw/Shanghai_T1DM/1006_1_20210209.xlsx'), PosixPath('../data/raw/Shanghai_T1DM/1006_2_20210303.xlsx'), PosixPath('../data/raw/Shanghai_T1DM/1007_0_20210726.xls'), PosixPath('../data/raw/Shanghai_T1DM/1008_0_20210713.xls'), PosixPath('../data/raw/Shanghai_T1DM/1009_0_20210803.xls'), PosixPath('../data/raw/Shanghai_T1DM/1010_0_20210915.xls'), PosixPath('../data/raw/Shanghai_T1DM/1011_0_20210622.xls'), PosixPath('../data/

In [3]:
# Look at the first 15 filenames so I can see the naming pattern
for f in files[:15]:
    print(f.relative_to(RAW))

.DS_Store
Shanghai_T1DM/1001_0_20210730.xlsx
Shanghai_T1DM/1002_0_20210504.xls
Shanghai_T1DM/1002_1_20210521.xls
Shanghai_T1DM/1002_2_20210909.xls
Shanghai_T1DM/1003_0_20210831.xls
Shanghai_T1DM/1004_0_20210425.xls
Shanghai_T1DM/1005_0_20210522.xls
Shanghai_T1DM/1006_0_20210114.xlsx
Shanghai_T1DM/1006_1_20210209.xlsx
Shanghai_T1DM/1006_2_20210303.xlsx
Shanghai_T1DM/1007_0_20210726.xls
Shanghai_T1DM/1008_0_20210713.xls
Shanghai_T1DM/1009_0_20210803.xls
Shanghai_T1DM/1010_0_20210915.xls


In [4]:
# Read the dataset's own notes first -> added by me
print((RAW / "note.txt").read_text())

This version is to add 188 CGM measurements on ShanghaiT2DM. 
In the "Shanghai_T2DM" folder, the file "2003_0_20210615" has extended rows from 586 to 670, with rows from 587 to 670 newly added.

In the "Shanghai_T2DM" folder, the file "2029_0_20210526" has extended rows from 742 to 846 with rows from 743 to 846 newly added. From line 743 to 755, the CGM values are missing because the CGM device has been changed to a new one. The readings of the new CGM device started from 2021/6/3 12:46.


“Shanghai_T2DM”文件夹中的“2003_0_20210615”文件从586行增加到670行，587~670行为新增数据
“Shanghai_T2DM”文件夹中的“2029_0_20210526”文件从742行增加到846行，743~846行为新增数据


2023/10/26


In [5]:
#get real cohort numbers -> added by me
import re

rows = []
for f in all_files:
    m = re.match(r"(\d+)_(\d+)_(\d{8})", f.name)
    if m:
        rows.append({"subject": m.group(1), "visit": int(m.group(2)),
                     "date": pd.to_datetime(m.group(3)),
                     "cohort": f.parent.name, "ext": f.suffix.lower()})

meta = pd.DataFrame(rows)
print("Files parsed:", len(meta), "of", len(all_files))
print("Distinct subjects:", meta["subject"].nunique())
print("\nBy cohort:")
print(meta.groupby("cohort").agg(files=("subject", "size"), subjects=("subject", "nunique")))
print("\nVisits per subject:")
print(meta.groupby("subject").size().value_counts().sort_index())

Files parsed: 125 of 132
Distinct subjects: 112

By cohort:
               files  subjects
cohort                        
Shanghai_T1DM     16        12
Shanghai_T2DM    109       100

Visits per subject:
1    102
2      7
3      3
Name: count, dtype: int64


**TODO — write down in a markdown cell below:**

- How many per-patient CGM files?
- What is the filename pattern? (Remember: patient IDs encode visit number —
  `2001_0` and `2001_1` are the *same person*, two visits.)
- How many `.xls` vs `.xlsx`?

### What I found

- 102 files for patients visited one, 7 files for patients visited twice, 3 files for patients visited 3 times.
- Patiend ID_visit_date
- xls: 101, xlsx: 26

## 2. The summary layer — one row per patient-visit

In [6]:
# Find the summary files. Adjust the pattern if the names differ.
summary_files = [f for f in files if "summary" in f.name.lower()]
print(summary_files)

[PosixPath('../data/raw/Shanghai_T1DM_Summary.xlsx'), PosixPath('../data/raw/Shanghai_T2DM_Summary.xlsx')]


In [ ]:
t1 = pd.read_excel(summary_files[0], na_values=["/"])   # check which is which before trusting this
t2 = pd.read_excel(summary_files[1], na_values=["/"])

print("T1 shape:", t1.shape)

print("T2 shape:", t2.shape)
t1.head()

T1 shape: ../data/raw/Shanghai_T2DM_Summary.xlsx (16, 33)
T2 shape: (109, 33)


,Patient Number,"Gender (Female=1, Male=2)",Age (years),Height (m),Weight (kg),BMI (kg/m2),Smoking History (pack year),Alcohol Drinking History (drinker/non-drinker),Type of Diabetes,Duration of Diabetes (years),Acute Diabetic Complications,Diabetic Macrovascular Complications,Diabetic Microvascular Complications,Comorbidities,Hypoglycemic Agents,Other Agents,Fasting Plasma Glucose (mg/dl),2-hour Postprandial Plasma Glucose (mg/dl),Fasting C-peptide (nmol/L),2-hour Postprandial C-peptide (nmol/L),Fasting Insulin (pmol/L),2-hour Postprandial Insulin (pmol/L),HbA1c (mmol/mol),Glycated Albumin (%),Total Cholesterol (mmol/L),Triglyceride (mmol/L),High-Density Lipoprotein Cholesterol (mmol/L),Low-Density Lipoprotein Cholesterol (mmol/L),Creatinine (umol/L),Estimated Glomerular Filtration Rate (ml/min/1.73m2),Uric Acid (mmol/L),Blood Urea Nitrogen (mmol/L),Hypoglycemia (yes/no)
0,1001_0_20210730,1,66,1.5,60.0,26.666667,0.0,non-drinker,T1DM,10.000000,none,"peripheral arterial disease, cerebrovascular d...","neuropathy, retinopathy","hypertension, osteoporosis, thyroid nodule, pu...","Humulin R, insulin detemir, acarbose","atorvastatin, aspirin, mecobalamin, calcitriol...",352.8,348.84,0.050,0.050,NaN,NaN,115.311,40.7,3.59,1.02,0.86,2.01,37.3,160.0,188.86,6.47,no
1,1002_0_20210504,2,68,1.7,63.0,21.799308,50.0,drinker,T1DM,26.000000,diabetic ketoacidosis,coronary heart disease,"neuropathy, retinopathy, nephropathy","hypertension, hypokalemia, hyperlipidemia, chr...","insulin aspart, insulin detemir","olmesartan medoxomil, benidipine, metoprolol, ...",181.8,258.84,0.016,0.016,543.38,754.71,69.405,19.6,4.78,2.20,0.93,3.28,66.8,109.0,342.57,6.05,yes
2,1002_1_20210521,2,68,1.7,67.0,23.183391,50.0,drinker,T1DM,26.000000,diabetic ketoacidosis,coronary heart disease,"neuropathy, retinopathy, nephropathy","hypertension, hypokalemia, hyperlipidemia, chr...","Novolin R, insulin glargine","olmesartan medoxomil, benidipine, metoprolol, ...",181.8,258.84,0.016,0.016,543.38,754.71,69.405,19.6,4.78,2.20,0.93,3.28,69.4,104.0,322.18,3.06,yes
3,1002_2_20210909,2,68,1.7,65.0,22.491349,50.0,drinker,T1DM,26.000000,none,coronary heart disease,"neuropathy, retinopathy, nephropathy","hypertension, hypokalemia, hyperlipidemia, chr...",Novolin R,"olmesartan medoxomil, benidipine, metoprolol, ...",237.6,NaN,0.016,0.016,78.37,74.39,72.684,25.1,3.49,1.82,0.84,1.83,63.7,115.0,342.34,6.21,yes
4,1003_0_20210831,2,37,1.9,60.0,16.620499,0.0,non-drinker,T1DM,0.083333,diabetic ketoacidosis,none,neuropathy,"leucopenia, hypokalemia, hepatic dysfunction","Novolin 50R, acarbose","mecobalamin, epalrestat, leucogen",120.6,248.76,0.100,0.120,NaN,NaN,121.869,46.6,5.61,1.14,1.08,3.95,49.6,174.0,93.39,1.85,yes


In [8]:
# Columns side by side — do the two summary sheets agree?
print("In T1 only:", sorted(set(t1.columns) - set(t2.columns)))
print("In T2 only:", sorted(set(t2.columns) - set(t1.columns)))
print("Shared:", len(set(t1.columns) & set(t2.columns)))

In T1 only: ['2-hour Postprandial Insulin (pmol/L)', 'Duration of Diabetes  (years)']
In T2 only: ['2-hour Postprandial insulin (pmol/L)', 'Duration of diabetes (years)']
Shared: 31


In [9]:
# Missingness per column, as a percentage. This is the single most useful
# table in the whole audit.
def missingness(df, name):
    out = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    out = out[out > 0]
    print(f"--- {name}: columns with missing values ---")
    print(out.to_string())
    print()
    return out

missingness(t1, "T1DM summary")
missingness(t2, "T2DM summary")

--- T1DM summary: columns with missing values ---
2-hour Postprandial Insulin (pmol/L)                      37.5
Fasting Insulin (pmol/L)                                  31.2
2-hour Postprandial Plasma Glucose (mg/dl)                12.5
Blood Urea Nitrogen (mmol/L)                               6.2
Uric Acid (mmol/L)                                         6.2
Estimated Glomerular Filtration Rate  (ml/min/1.73m2)      6.2
Creatinine (umol/L)                                        6.2
Glycated Albumin (%)                                       6.2
HbA1c (mmol/mol)                                           6.2
2-hour Postprandial C-peptide (nmol/L)                     6.2
Fasting C-peptide (nmol/L)                                 6.2

--- T2DM summary: columns with missing values ---
2-hour Postprandial insulin (pmol/L)                      50.5
2-hour Postprandial C-peptide (nmol/L)                    41.3
Fasting Insulin (pmol/L)                                  30.3
Blood Urea Nitrog

2-hour Postprandial insulin (pmol/L)                      50.5
2-hour Postprandial C-peptide (nmol/L)                    41.3
Fasting Insulin (pmol/L)                                  30.3
Blood Urea Nitrogen (mmol/L)                              24.8
Uric Acid (mmol/L)                                        22.0
Estimated Glomerular Filtration Rate  (ml/min/1.73m2)     22.0
Creatinine (umol/L)                                       21.1
Glycated Albumin (%)                                      21.1
Fasting C-peptide (nmol/L)                                21.1
2-hour Postprandial Plasma Glucose (mg/dl)                20.2
Total Cholesterol (mmol/L)                                20.2
Low-Density Lipoprotein Cholesterol (mmol/L)              16.5
High-Density Lipoprotein Cholesterol (mmol/L)             16.5
Triglyceride (mmol/L)                                     16.5
HbA1c (mmol/mol)                                           7.3
Fasting Plasma Glucose (mg/dl)                         

In [10]:
# The outcome variable. Find the hypoglycemia column and check its balance.
# TODO: replace 'Hypoglycemia' with the actual column name once you have seen it.
hypo_cols = [c for c in t2.columns if "hypo" in str(c).lower()]
print("Candidate columns:", hypo_cols)
#print(t2["Hypoglycemic Agents"]) -> to check the data stored in this col
#print(t2["Hypoglycemia (yes/no)"]) -> to check the data stored in this col

# Once identified:
hypo_cols = "Hypoglycemia (yes/no)"
print(t2[hypo_cols].value_counts(dropna=False))
print(t1[hypo_cols].value_counts(dropna=False))

Candidate columns: ['Hypoglycemic Agents', 'Hypoglycemia (yes/no)']
Hypoglycemia (yes/no)
no     99
yes    10
Name: count, dtype: int64
Hypoglycemia (yes/no)
yes    14
no      2
Name: count, dtype: int64


**TODO — answer in the cell below:**

1. How many summary files, and what are they called?
2. What is the exact name of the hypoglycemia column, and how is it coded?
3. What is the class balance? (This number decides your evaluation metric in Phase 4.)
4. Which clinical columns are so sparse they are unusable?
5. Are units stated anywhere, or assumed? (HbA1c %, glucose mg/dl vs mmol/l — getting this wrong silently ruins every downstream metric.)

**Answers:**
1. 2,  Shanghai_T1DM_Summary.xlsx, Shanghai_T2DM_Summary.xlsx
2. There  are 2: ‘Hypoglycemic Agents' and ’Hypoglycemia (yes/no)’, first with the name of the medicine, second with yes/no
3. Class balance: T1DM: 12.5% no/87.5% yes, T2DM: 90.8% no/9.2% yes, Combine: 80.8% no/19.2% yes
4. The 2-hour postprandial insulin and C-peptide columns are the clear standouts — 37–50% missing across both cohorts That's high enough to call "unusable as a reliable feature" rather than just "somewhat sparse." 
5. Units are documented in header for every column, however the unit of HBA1C is documented in mmol/L instead of % which is more commonly used.

In [11]:
missingness_series = missingness(t2, "T2DM summary")
threshold = 30  # % missing
sparse_cols = missingness_series[missingness_series > threshold].index.tolist()
print("Columns too sparse to use as features:", sparse_cols)
# check how many POSITIVE cases you'd lose if you dropped rows missing this column
for col in sparse_cols:
    lost = t2[t2[col].isna()][hypo_cols].value_counts()
    print(col, "->", dict(lost))

--- T2DM summary: columns with missing values ---
2-hour Postprandial insulin (pmol/L)                      50.5
2-hour Postprandial C-peptide (nmol/L)                    41.3
Fasting Insulin (pmol/L)                                  30.3
Blood Urea Nitrogen (mmol/L)                              24.8
Uric Acid (mmol/L)                                        22.0
Estimated Glomerular Filtration Rate  (ml/min/1.73m2)     22.0
Creatinine (umol/L)                                       21.1
Glycated Albumin (%)                                      21.1
Fasting C-peptide (nmol/L)                                21.1
2-hour Postprandial Plasma Glucose (mg/dl)                20.2
Total Cholesterol (mmol/L)                                20.2
Low-Density Lipoprotein Cholesterol (mmol/L)              16.5
High-Density Lipoprotein Cholesterol (mmol/L)             16.5
Triglyceride (mmol/L)                                     16.5
HbA1c (mmol/mol)                                           7.3
Fasti

### What I found

To answer the 4th questions: Which clinical columns are so sparse they are unusable?
I need to fix the missingness funtion -> it didn't catch the cells holds "/"
add: na_values=["/"]

## 3. The CGM layer — one patient file

In [ ]:
# Pick one T1 and one T2 patient file (not a summary) and open them.
cgm_files = [f for f in files if "summary" not in f.name.lower()
             and f.suffix.lower() in (".xls", ".xlsx")]
print(f"{len(cgm_files)} CGM files")
print(cgm_files[0].name)

p = pd.read_excel(cgm_files[0])
print(p.shape)
p.head(20)

125 CGM files
1001_0_20210730.xlsx
../data/raw/Shanghai_T1DM/1001_0_20210730.xlsx
(658, 11)


,Date,CGM (mg / dl),CBG (mg / dl),Blood Ketone (mmol / L),Dietary intake,饮食,Insulin dose - s.c.,Non-insulin hypoglycemic agents,"CSII - bolus insulin (Novolin R, IU)","CSII - basal insulin (Novolin R, IU / H)",Insulin dose - i.v.
0,2021-07-30 16:43:00,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,NaN
1,2021-07-30 16:58:00,124.2,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN
2,2021-07-30 17:13:00,129.6,NaN,NaN,data not available,未记录,NaN,NaN,NaN,NaN,NaN
3,2021-07-30 17:28:00,142.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-07-30 17:43:00,156.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2021-07-30 17:58:00,162.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2021-07-30 18:13:00,163.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2021-07-30 18:28:00,165.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2021-07-30 18:43:00,169.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2021-07-30 18:58:00,167.4,192.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Column names exactly as they appear — including any Chinese-language duplicates
for c in p.columns:
    print(repr(c))

'Date'
'CGM (mg / dl)'
'CBG (mg / dl)'
'Blood Ketone (mmol / L)'
'Dietary intake'
'饮食'
'Insulin dose - s.c.'
'Non-insulin hypoglycemic agents'
'CSII - bolus insulin (Novolin R, IU)'
'CSII - basal insulin (Novolin R, IU / H)'
'Insulin dose - i.v.'


In [14]:
# Timestamps: are they regular?
# TODO: replace 'Date' with the real timestamp column name.
ts_col = "Date"

p[ts_col] = pd.to_datetime(p[ts_col], errors="coerce")
gaps = p[ts_col].diff().dt.total_seconds() / 60

print("Recording spans:", p[ts_col].min(), "to", p[ts_col].max())
print("Duration (days):", round((p[ts_col].max() - p[ts_col].min()).total_seconds() / 86400, 2))
print("\nMinutes between readings:")
print(gaps.value_counts().head(10))
print("\nGaps longer than 30 min:", (gaps > 30).sum())

Recording spans: 2021-07-30 16:43:00 to 2021-08-06 12:58:00
Duration (days): 6.84

Minutes between readings:
Date
15.0    657
Name: count, dtype: int64

Gaps longer than 30 min: 0


In [15]:
# The known problem case: patient 2029_0 has missing CGM values from a device swap,
# readings resume 2021-06-03 12:46. Find that file and look at it directly.
target = [f for f in cgm_files if "2029" in f.name]
print(target)

# TODO: open it, locate the gap, and record exactly how long it is.

[PosixPath('../data/raw/Shanghai_T2DM/2029_0_20210526.xls')]


**TODO — answer:**

- What is the sampling interval, and is it consistent?
- How long is the 2029_0 gap, in hours?
- What fraction of *expected* readings are actually present, for a few patients?
- Are there appended junk rows at the bottom of any file? (Check `.tail(20)`.)

### What I found

*(replace this text)*

## 4. Scale check — loop over all files

In [16]:
# Do NOT build the cleaning pipeline here. Just collect shapes and basic facts
# so you know what you are dealing with. This may take a minute.

records = []
for f in cgm_files:
    try:
        df = pd.read_excel(f)
        records.append({
            "file": f.name,
            "ext": f.suffix.lower(),
            "rows": len(df),
            "cols": df.shape[1],
            "colnames": tuple(df.columns),
        })
    except Exception as e:
        records.append({"file": f.name, "ext": f.suffix.lower(),
                        "rows": None, "cols": None, "colnames": f"ERROR: {e}"})

inv = pd.DataFrame(records)
print(inv[["rows", "cols"]].describe())
inv.head()

              rows   cols
count   125.000000  125.0
mean   1025.360000   11.0
std     334.208994    0.0
min     247.000000   11.0
25%     751.000000   11.0
50%    1217.000000   11.0
75%    1330.000000   11.0
max    1339.000000   11.0


,file,ext,rows,cols,colnames
0,1001_0_20210730.xlsx,.xlsx,658,11,"(Date, CGM (mg / dl), CBG (mg / dl), Blood Ket..."
1,1002_0_20210504.xls,.xls,948,11,"(Date, CGM (mg / dl), CBG (mg / dl), Blood Ket..."
2,1002_1_20210521.xls,.xls,933,11,"(Date, CGM (mg / dl), CBG (mg / dl), Blood Ket..."
3,1002_2_20210909.xls,.xls,357,11,"(Date, CGM (mg / dl), CBG (mg / dl), Blood Ket..."
4,1003_0_20210831.xls,.xls,1339,11,"(Date, CGM (mg / dl), CBG (mg / dl), Blood Ket..."


In [17]:
# Do all files have the same columns? If not, that is a Phase 1 problem to solve.
schema_counts = inv["colnames"].value_counts()
print(f"{len(schema_counts)} distinct column layouts across {len(inv)} files\n")
for schema, n in schema_counts.items():
    print(f"{n} files:")
    print("   ", schema if isinstance(schema, str) else list(schema))
    print()

5 distinct column layouts across 125 files

119 files:
    ['Date', 'CGM (mg / dl)', 'CBG (mg / dl)', 'Blood Ketone (mmol / L)', 'Dietary intake', '饮食', 'Insulin dose - s.c.', 'Non-insulin hypoglycemic agents', 'CSII - bolus insulin (Novolin R, IU)', 'CSII - basal insulin (Novolin R, IU / H)', 'Insulin dose - i.v.']

2 files:
    ['Date', 'CGM (mg / dl)', 'CBG (mg / dl)', 'Blood Ketone (mmol / L)', 'Dietary intake', '进食量', 'Insulin dose - s.c.', 'Non-insulin hypoglycemic agents', 'CSII - bolus insulin (Novolin R, IU)', 'CSII - basal insulin (Novolin R, IU / H)', 'Insulin dose - i.v.']

2 files:
    ['Date', 'CGM ', 'CBG ', 'Blood Ketone ', 'Dietary intake', '饮食', 'Insulin dose - s.c.', 'Non-insulin hypoglycemic agents', 'CSII - bolus insulin ', 'CSII - basal insulin ', 'Insulin dose - i.v.']

1 files:
    ['Date', 'CGM (mg / dl)', 'CBG (mg / dl)', 'Blood Ketone (mmol / L)', 'Dietary intake', '饮食', 'Insulin dose - s.c.', 'Non-insulin hypoglycemic agents', 'CSII - bolus insulin (Novolin 

In [18]:
# Any files that failed to open?
print(inv[inv["rows"].isna()])

Empty DataFrame
Columns: [file, ext, rows, cols, colnames]
Index: []


## 5. Cohort summary

**TODO — fill these in. These numbers go straight into the README.**

| Fact | Value |
|---|---|
| Total patients | |
| Total patient-visits | |
| Patients with repeat visits | |
| T1DM / T2DM split | |
| Median recording days per patient | |
| Hypoglycemia positive rate | |
| Distinct column layouts | |
| Files that failed to open | |

## 6. Decisions to carry into Phase 1

Every line here becomes a row in `docs/data-decisions.md`, with a rationale.

**TODO — list them. Starter set, confirm or change each:**

1. Glucose gaps will be **flagged, not imputed** — because imputing glucose invents
   physiology, and it silently changes every variability metric.
2. Resample to a regular 15-minute grid — because Time in Range computed on irregular
   timestamps is a different number, and it has to be reproducible.
3. Drop / rename the duplicate Chinese-language column — decide which, and say why.
4. Repeat visits are **not independent observations** — this constrains cross-validation
   in Phase 4.
5. ...

## What surprised me

*(The most valuable cell in this notebook. Write honestly — the thing you did not
expect is usually the thing worth talking about in an interview.)*